<a href="https://colab.research.google.com/github/Ayu-sshhhhh/Food-Vision-Classifier/blob/main/Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


# Importing dataset

10% of 10-food-classes from Food 101 - https://www.kaggle.com/dansbecker/food-101

https://github.com/mrdbourke/tensorflow-deep-learning/blob/main/04_transfer_learning_in_tensorflow_part_1_feature_extraction.ipynb

In [ ]:
import zipfile
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
zip_ref = zipfile.ZipFile("10_food_classes_10_percent.zip")
zip_ref.extractall()
zip_ref.close()

--2026-04-21 14:15:06--  https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.152.207, 108.177.121.207, 173.194.206.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.152.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 168546183 (161M) [application/zip]
Saving to: ‘10_food_classes_10_percent.zip.2’

10_food_classes_10_ 100%[===================>] 160.74M   289MB/s    in 0.6s    

2026-04-21 14:15:07 (289 MB/s) - ‘10_food_classes_10_percent.zip.2’ saved [168546183/168546183]



# Glimpse of dataset

In [ ]:
import os
for dirpath, dirnames, filenames in os.walk("10_food_classes_10_percent"):
  print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath} ")

There are 2 directories and 0 images in '10_food_classes_10_percent 
There are 10 directories and 0 images in '10_food_classes_10_percent/test 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/grilled_salmon 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/ice_cream 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/sushi 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/chicken_wings 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/pizza 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/steak 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/hamburger 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/fried_rice 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/ramen 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/chicken_curry

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SHAPE = (224,224)
BATCH_SIZE = 32
EPOCHS = 5

train_dir = "10_food_classes_10_percent/train/"
test_dir = "10_food_classes_10_percent/test/"

train_datagen = ImageDataGenerator(rescale=1/255.)
test_datagen = ImageDataGenerator(rescale=1/255.)
print("Training Images : ")
train_data_10_percent = train_datagen.flow_from_directory(train_dir,
                                                          target_size = IMAGE_SHAPE,
                                                          batch_size = BATCH_SIZE,
                                                          class_mode = 'categorical')
print("Testing Images : ")
test_data_10_percent = test_datagen.flow_from_directory(test_dir,
                                                        target_size = IMAGE_SHAPE,
                                                        batch_size = BATCH_SIZE,
                                                        class_mode = 'categorical')

Training Images : 
Found 750 images belonging to 10 classes.
Testing Images : 
Found 2500 images belonging to 10 classes.


# Setting up callbacks

In [ ]:
# creating Tensorboard callbacks ( and functionizing it)
import datetime
def create_tensorboard_callback(dir_name, experiment_name):
  log_dir = dir_name +"/" + experiment_name +"/"+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
  tensorboard_callback = tf.keras.callbacks.Tensorboard(log_dir=log_dir)
  print(f"Saving TensorBoard log files to : {log_dir}")
  return tensorboard_callback

We found the following feature vector model link:
https://www.kaggle.com/models/tensorflow/resnet-50/TensorFlow2/feature-vector/1

In [ ]:
rest_net = "https://www.kaggle.com/models/tensorflow/resnet-50/TensorFlow2/feature-vector/1"

In [ ]:
import os
#os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers

In [ ]:
IMAGE_SHAPE + (3,)

(224, 224, 3)

In [ ]:
def create_model(model_url, num_classes=10):
  """
  takes a tensorflow Hub URL and creates keras sequential model with it.
   Args :
   model_url (str) : A tensorflow hub feature extraction URL.
   num_classes (int) : number of output neurons in the output layers.
   should be equal to the numbre of target classes, default 10

   Returns :
   an uncompiled Keras Sequential Model with model_url as feature extractor layer and Dense output layer with num_classes output features.
   """

  feature_extractor_layer = hub.KerasLayer(model_url,
                                            trainable = False,
                                            name = "feature_extractor_layer",
                                            input_shape = IMAGE_SHAPE+(3,))

  model = tf.keras.Sequential([
       feature_extractor_layer,
       layers.Dense(num_classes, activation = "softmax", name = "output_layer")
   ])

  return model

### Create RESTNet Tensorflow Hub Feature Extraction Model

In [ ]:
restnet_model = create_model(rest_net,
                             num_classes = train_data_10_percent.num_classes)


ValueError: Only instances of `keras.Layer` can be added to a Sequential model. Received: <tensorflow_hub.keras_layer.KerasLayer object at 0x78b5f4a23c50> (of type <class 'tensorflow_hub.keras_layer.KerasLayer'>)

In [ ]:
restnet_model.summary()

In [ ]:
restnet_model.compile(loss="categorical_crossentropy",
                      optimizer = "Adam",
                      metrics=["accuracy"])

### Fitting our RestNet Model to the data

In [ ]:
def create_tensorboard_callback(dir_name, experiment_name):
  log_dir = dir_name + "/" + experiment_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
  # Note the capital 'B' in TensorBoard below
  tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir)
  print(f"Saving TensorBoard log files to: {log_dir}")
  return tensorboard_callback

In [ ]:
restnet_history = restnet_model.fit(train_data_10_percent,
                                    epochs=5,
                                    steps_per_epoch=len(train_data_10_percent),
                                    validation_data=test_data_10_percent,
                                    validation_steps=len(test_data_10_percent),
                                    callbacks=[create_tensorboard_callback(dir_name="tensorflow_hub",
                                                                           experiment_name="restnet50V1")])